### Universidade Federal da Bahia
### Escola Politécnica
### Programa de Pós-Graduação em Engenharia Industrial
### Discente: Ghabriel Anton Gomes de Sá
### Orientadores: Marcelo Embiruçu e Cristiano Fontes

<center><h1>UMA ABORDAGEM COMBINANDO UMA ESTIMATIVA DE PESOS INICIAIS BASEADA EM LINEARIZAÇÃO E ALGORITMO CONSTRUTIVISTA PARA REDES NEURAIS DE PROPAGAÇÃO DIRETA COM UMA ÚNICA CAMADA OCULTA APLICADAS À IDENTIFIÇÃO DE MODELOS COM MÚLTIPLAS ENTRADAS E ÚNICA SAÍDA EM PROBLEMAS DE CLASSIFICAÇÃO BINÁRIA</center></h1>

#### <p style='text-align: justify;'>O Método para a obtenção dos pesos iniciais propõe que: (i) seja realizada a linearização, via série de Taylor, do modelo não-linear a ser utilizado para a modelagem do conjunto de dados (o modelo consiste em uma rede neural com p entradas, uma única camada oculta e um neurônio na camada de saída), onde o ponto de equilíbrio para a linearização é a média das suas respectivas variáveis (x,y); (ii) seja realizada a Regressão Linear Múltipla, via Método dos Mínimos Quadrados, do conjunto de dados a ser modelado; (iii) através da comparação matemática entre (i) e (ii), obtenha-se os pesos que conectam os neurônios de entrada ao primeiro neurônio oculto da rede. 
#### <p style='text-align: justify;'>Em seguida, a rede continua a aumentar o número de neurônios ocultos a partir de uma abordagem construtivista. Toda a rede é treinada a cada adição de um novo neurônio oculto, sendo que os pesos obtidos no final de um treinamento são utilizados como pesos iniciais para o treinamento subsequente. Os pesos iniciais do neurônio oculto a ser adicionado são obtidos a partir do método de Bengio e Glorot (2010), conhecido como método de Xavier. Comparou-se o método proposto (WILCA-BC) com outros dois métodos (RIXM e ELM).

## Importação de pacotes

In [ ]:
# Manipulação de dados:
import pandas as pd
from sklearn import preprocessing
from sklearn.model_selection import train_test_split

# Resolução de sistema de eq. não-lineares:
import statsmodels.api as sm
from scipy.optimize import fsolve

# Álgebra linear:
import numpy as np
import math
from scipy.linalg import pinv2 #Usado no ELM

# Cálculo da acurácia:
from sklearn.metrics import accuracy_score

# Contagem de tempo:
import time

# Criação e plotagem de gráficos:
import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib.ticker as mtick

# BCE:
import tensorflow as tf

## Importação do banco de dados
<p style='text-align: justify;'>Para que a importação do banco de dados seja feita da maneira correta, é necessário que o banco de dados a ser importado esteja de acordo com o padrão apresentado a seguir e também que ele esteja localizado na mesma pasta que esse código. A princípio, esse código está implementado em um arquivo .ipynb.</p>

O banco de dados deve apresentar, obrigatoriamente, as seguintes características:

-Deve ser um arquivo do tipo .csv;

-Os dados devem estar delimitados por ponto e vírgula (;), ou seja, devem estar presentes em diferentes células da planilha Excel;

-As variáveis de entradas são todas as colunas, exceto a última (que trata-se da variável de saída);

-Não deveve possuir cabeçalho (linha com nome dos atributos).

In [ ]:
# Pedindo informações do conjunto de dados (nome do conjunto e se ele possui cabeçalho ou não):
filename = input("Digite o nome do arquivo .csv que contém o banco de dados:")
headdecider = input(
    "O conjunto de dados possui header (nome dos atributos)? Responda com S ou N.")

# Abrindo conjunto de dados:
if headdecider in ['S']:
    data = pd.read_csv(filename+'.csv', sep=';')
elif headdecider in ['N']:
    data = pd.read_csv(filename+'.csv', header=None, sep=';')
else:
    print("Você digitou um comando incorreto. Os comandos possíveis são 'S' ou 'N'")

## Pré-processamento de dados

#### • Obtendo primeiras informações sobre o conjunto de dados:

In [ ]:
# Mostrando cinco primeiras linhas de dados:
data.head()

In [ ]:
# Mostrando informações sobre os atributos:
data.info()

In [ ]:
# Descrição estatística dos dados:
data.describe()

#### • Lidando com os dados faltantes:

In [ ]:
# Verificando a existência de dados faltantes:
data.isna().sum()

In [ ]:
# Deletando dados faltantes:
data.dropna(inplace=True)

In [ ]:
# Missing values também podem estar representadas através do número 0. Por conta disso, deve-se verificar se os zeros são missing values ou se são
# inerentes ao atributo:
num_zeros = (data == 0).sum()
print('Quantidade de valores nulos:\n', num_zeros)

#### • Reduzindo a escala dos dados:
Essa etapa garante que os dados estejam entre 0 e 1.

In [ ]:
# Cria uma array a partir dos dados:
z = data.values

# Atribuindo nova escala os dados:
min_max_scaler = preprocessing.MinMaxScaler()
data_normalized = min_max_scaler.fit_transform(z)

In [ ]:
print("Dados normalizados:\n", data_normalized)

#### • Delimitação de entrada e saída:

In [ ]:
# Delimitando:
inputs_all = data_normalized[:, :-1]
targets_all = data_normalized[:, -1]

# Atribuindo valor ao número de atributos (número de colunas dos dados de entrada):
n_inputs = inputs_all.shape[1]

In [ ]:
# Exibindo valores em tela:
print("Inputs:\n", inputs_all)
print("Outputs:\n", targets_all)
print("Nº de atributos:\n", n_inputs)

## Criação dos Modelos
<p style='text-align: justify;'>Os modelos são do tipo SFNN e treinados de acordo com o algoritmo de retropropagação (ARP); exceto o Método 3, que utiliza o ELM.</p>

## Definindo função para cálculo de BCE
BCE é a função de perda, utilizado para a otimização do ARP.

In [ ]:
bce = tf.keras.losses.BinaryCrossentropy()

## MÉTODO 1
Modelo SFNN, treinado via ARP, considerando os pesos iniciais (e bias) do primeiro neurônio oculto obtidos a partir de Linearização e Análise via Série de Taylor e pesos dos demais neurônios ocultos, bem como do neurônio de saída obtidos a partir do método de Xavier, preservando os pesos dos neurônios já aprendidos e utilizando-os como estimativas iniciais para o próximo treinamento.

### Aplicação de Regressão Linear no conjunto de dados
<p style='text-align: justify;'>A regressão é feita utilizando o módulo statsmodels.api. O método utilizado para a regressão é o Método dos Mínimos Quadrados.</p>

In [ ]:
# Adiciona a coluna das constantes
inputs_all_cte = sm.add_constant(inputs_all, prepend=True)
res = sm.OLS(targets_all, inputs_all_cte).fit()  # Cria e ajusta o modelo
print(res.summary())  # Imprime os resultados

### Resolução do sistema de equações proposto para obter os parâmetros iniciais da rede
<p style='text-align: justify;'>A resolução é feita utilizando o módulo scipy.optimize.fsolve. O módulo retorna as raízes das equações (não lineares) definidas por f(x)= 0, dada uma estimativa inicial.</p>

In [ ]:
# Definindo constantes:
x_M = []
for i in range(n_inputs):
    x_M.append(np.mean(inputs_all[:, i]))
k = []
for i in range(n_inputs):
    k.append(res.params[i+1])
b = res.params[0]

In [ ]:
# Definindo chutes iniciais para beta e wi como valores aleatórios entre 0 e 1:
x0 = np.random.rand(n_inputs+1, 1)

In [ ]:
# Definindo função sigmoide e sua derivada:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def dsigmoid(x):
    z = sigmoid(x)
    return z * (1 - z)

In [ ]:
# Inicializando variáveis wi e beta:
w = [0 for i in range(n_inputs)]
beta = 0

In [ ]:
# Definindo sistema de equações a ser resolvido:
def f(x):
    w, beta = x[:-1], x[-1]
    f = [sigmoid(np.dot(w, x_M)+beta)-(sigmoid(np.dot(w, x_M)+beta) -
                                       (sigmoid(np.dot(w, x_M)+beta))**2)*np.dot(w, x_M)-b]
    for i in range(n_inputs):
        f.append((sigmoid(np.dot(w, x_M)+beta) -
                 (sigmoid(np.dot(w, x_M)+beta))**2)*w[i]-k[i])
    return f

In [ ]:
# Resolvendo o sistema e imprimindo resultados (pesos e bias iniciais):
result = fsolve(f, x0)
print(result)

### Criando Método 1:

In [ ]:
class Method1:
    def __init__(self, x, y, n):

        # vetor de saídas preditas. Inicialmente um vetor de zeros.
        self.Yh = np.zeros((1, y.shape[1]))
        self.n = n  # número de neurônios ocultos.
        # número de neurônios em cada camada (entrada, intermediária e saída).
        self.dims = [x.shape[0], n, 1]
        # um dicionário que conterá os parâmetros W e b entre as camadas da rede (entrada-intermediária e intermediária-saída); ou seja: W1, b1, W2 e b2.
        self.param = {}
        self.ch = {}  # parâmetro intermediário de cálculo.
        self.grad = {}  # parâmetro intermediário de cálculo.
        self.loss = []  # erro de treinamento (BCE).
        self.lr = 0.1  # taxa de aprendizagem.
        # número de observações do conjunto de treinamento.
        self.sam = y.shape[1]

    def nInit(self):  # inicializa os pesos e os bias. Nesse caso, os pesos e os bias do primeiro neurônio oculto corresponde aos pesos encontrados pelo processo acima.

        self.param['W1'] = np.random.uniform(-math.sqrt(6)/math.sqrt(
            self.dims[0]+1), math.sqrt(6)/math.sqrt(self.dims[0]+1), (self.dims[1], self.dims[0]))
        self.param['W1'][0] = result[:-1]
        self.param['b1'] = np.random.uniform(-math.sqrt(6)/math.sqrt(
            self.dims[0]+1), math.sqrt(6)/math.sqrt(self.dims[0]+1), (self.dims[1], 1))
        self.param['b1'][0] = result[-1]
        self.param['W2'] = np.random.uniform(-math.sqrt(6)/math.sqrt(
            self.dims[1]+1), math.sqrt(6)/math.sqrt(self.dims[1]+1), (self.dims[2], self.dims[1]))
        self.param['b2'] = np.random.uniform(-math.sqrt(6)/math.sqrt(
            self.dims[1]+1), math.sqrt(6)/math.sqrt(self.dims[1]+1), (self.dims[2], 1))
        return

# Definindo função para cálculo de erro quadrático:
    def nloss(self):
        loss = bce(y, self.Yh)
        return loss

    def forward(self, x):
        Z1 = self.param['W1'].dot(x) + self.param['b1']
        A1 = sigmoid(Z1)
        self.ch['Z1'], self.ch['A1'] = Z1, A1
        Z2 = self.param['W2'].dot(A1) + self.param['b2']
        A2 = sigmoid(Z2)
        self.ch['Z2'], self.ch['A2'] = Z2, A2
        self.Yh = A2
# Acabamos de processar nossos dados de entrada pela rede e produzimos Yh, uma saída.
# O próximo passo é descobrir o quão bom foi o nosso resultado.
# Para isso, podemos comparar a saída que produzimos com a saída que deveríamos ter obtido: Yh e Y. Para isso, aplicaremos a função de erro:
        loss = self.nloss()
        return self.Yh, loss

    def backward(self):
        dLoss_Yh = - (y-self.Yh)

        dLoss_Z2 = dLoss_Yh * dsigmoid(self.ch['Z2'])
        dLoss_A1 = np.dot(self.param["W2"].T, dLoss_Z2)
        dLoss_W2 = 1/self.ch['A1'].shape[1] * np.dot(dLoss_Z2, self.ch['A1'].T)
        dLoss_b2 = 1/self.ch['A1'].shape[1] * \
            np.dot(dLoss_Z2, np.ones([dLoss_Z2.shape[1], 1]))

        dLoss_Z1 = dLoss_A1 * dsigmoid(self.ch['Z1'])
        dLoss_A0 = np.dot(self.param["W1"].T, dLoss_Z1)
        dLoss_W1 = 1/x.shape[1] * np.dot(dLoss_Z1, x.T)
        dLoss_b1 = 1/x.shape[1] * \
            np.dot(dLoss_Z1, np.ones([dLoss_Z1.shape[1], 1]))

        self.param["W1"] = self.param["W1"] - self.lr * dLoss_W1
        self.param["b1"] = self.param["b1"] - self.lr * dLoss_b1
        self.param["W2"] = self.param["W2"] - self.lr * dLoss_W2
        self.param["b2"] = self.param["b2"] - self.lr * dLoss_b2

    def train(self, X, Y, iter=1501):
        for i in range(0, iter):
            Yh, loss = self.forward(x)
            self.backward()
        self.loss.append(loss)
        return

    def test(self, x, y):
        pred, loss = self.forward(x)
        return loss

### Executando Método 1:

In [ ]:
# Inicializando contagem do tempo:
ini = time.time()

# Abrindo listas para Acurácia MAX (treino e teste):
acc_train_max = []
acc_test_max = []

# Inicializando loop de experimentos:
for l in range(1, 51):
    np.random.seed(l)
    train_inputs, test_inputs, train_targets, test_targets = train_test_split(
        inputs_all, targets_all, test_size=0.20, random_state=l)
    print("Experimento", l,)
    # Abrindo listas para BCE e Acurácia (treino e teste):
    bce_train = []
    bce_test = []
    acc_train = []
    acc_test = []

    # Abrindo lista para pesos:
    params_models = []

    # Sinalização, em tela, início de rede com 1 neurônio oculto:
    print("Initializing Network with 1 hidden node:")

    # Definindo entrada e saída de treino:
    x = train_inputs.T
    y = np.ndarray.tolist(train_targets)
    y = [y]
    y = np.asarray(y)

    # Criando modelo:
    Modelo = Method1(x, y, 1)

    # Inicializando pesos:
    Modelo.nInit()

    # Treinando modelo:
    Modelo.train(x, y)

    # BCE de treino:
    bce_train.append(Modelo.loss)
    # print("BCE de treino:",Modelo.loss)

    # Acurácia de treino:
    for n in range(0, Modelo.Yh.shape[1]):
        if Modelo.Yh[0][n] > 0.5:
            Modelo.Yh[0][n] = 1
        else:
            Modelo.Yh[0][n] = 0
    Yh = [item for elem in Modelo.Yh for item in elem]
    y = np.ndarray.tolist(train_targets)
    acc_train.append(accuracy_score(y, Yh))
    print("Train Accuracy:", accuracy_score(y, Yh))

    # Registrando pesos já aprendidos:
    params_models.append(Modelo.param)

    # Definindo entrada e saída de teste:
    x = test_inputs.T
    y = np.ndarray.tolist(test_targets)
    y = [y]
    y = np.asarray(y)

    # Testando modelo e calculando BCE de teste:
    bce_test.append(Modelo.test(x, y))

    # Acurácia de teste:
    for n in range(0, Modelo.Yh.shape[1]):
        if Modelo.Yh[0][n] > 0.5:
            Modelo.Yh[0][n] = 1
        else:
            Modelo.Yh[0][n] = 0
    Yh = [item for elem in Modelo.Yh for item in elem]
    y = np.ndarray.tolist(test_targets)
    acc_test.append(accuracy_score(y, Yh))
    print("Test Accuracy:", accuracy_score(y, Yh))

    # Inicializando loop para demais neurônios ocultos:
    for n in range(2, 251):

      # Sinalização, em tela, de início de rede com n neurônios ocultos:
        print("Network with", n, "hidden nodes:")

        # Definindo entrada e saída de treino:
        x = train_inputs.T
        y = np.ndarray.tolist(train_targets)
        y = [y]
        y = np.asarray(y)

        # Criando Modelo:
        Modelo = Method1(x, y, n)

        # Inicializando novos pesos e utilizando pesos do treinamento anterior:
        Modelo.nInit()
        Modelo.param['W1'][:-1] = params_models[-1]['W1']
        Modelo.param['b1'][:-1] = params_models[-1]['b1']
        Modelo.param['W2'][-1][:-1] = params_models[-1]['W2']
        Modelo.param['b2'] = params_models[-1]['b2']

        # Treinando modelo:
        Modelo.train(x, y)

        # Registro e exibição de BCE de treino:
        bce_train.append(Modelo.loss)
        print("Train BCE:", Modelo.loss)

        # Registro e exibição de acurácia de treino:
        for n in range(0, Modelo.Yh.shape[1]):
            if Modelo.Yh[0][n] > 0.5:
                Modelo.Yh[0][n] = 1
            else:
                Modelo.Yh[0][n] = 0
        Yh = [item for elem in Modelo.Yh for item in elem]
        y = np.ndarray.tolist(train_targets)
        acc_train.append(accuracy_score(y, Yh))
        print("Train Accuracy:", accuracy_score(y, Yh))

        # Registrando pesos já aprendidos:
        params_models.append(Modelo.param)

        # Definindo entrada e saída de teste:
        x = test_inputs.T
        y = np.ndarray.tolist(test_targets)
        y = [y]
        y = np.asarray(y)

        # Testando modelo e calculando BCE de teste:
        bce_test.append(Modelo.test(x, y))

        # Registro e exibição de Acurácia de teste:
        for n in range(0, Modelo.Yh.shape[1]):
            if Modelo.Yh[0][n] > 0.5:
                Modelo.Yh[0][n] = 1
            else:
                Modelo.Yh[0][n] = 0
        Yh = [item for elem in Modelo.Yh for item in elem]
        y = np.ndarray.tolist(test_targets)
        acc_test.append(accuracy_score(y, Yh))
        print("Test Accuracy:", accuracy_score(y, Yh))
    # Registrando Acurácia máximo de treino e teste:
    acc_train_max.append(acc_train[acc_test.index(max(acc_test))])
    acc_test_max.append(max(acc_test))

In [ ]:
# Finalizando e exibindo contagem de tempo de processamento:
fim = time.time()
print("Training and Testing time:", fim-ini, "(s)")

In [ ]:
acc_train1 = acc_train_max
acc_test1 = acc_test_max

In [ ]:
# Exibindo em tela Acurácia de teste máxima médio de treino:
print("Mean of Max Train Accuracy:", np.mean(acc_train_max))
print("Standard Deviation of Max Train Accuracy:", np.std(acc_train_max))

In [ ]:
# Exibindo em tela Acurácia de teste máximo médio de teste:
print("Mean of Max Test Accuracy:", np.mean(acc_test_max))
print("Standard Deviation of Max Test Accuracy:", np.std(acc_test_max))

### MÉTODO 2
Modelo SFNN considerando os pesos iniciais e bias do neurônio oculto e do neurônio de saída obtidos a partir do método de Xavier, desconsiderando os pesos dos neurônios já aprendidos.

### Criando Método 2:

In [ ]:
class Method2:
    def __init__(self, x, y, n):

        # vetor de saídas preditas. Inicialmente um vetor de zeros.
        self.Yh = np.zeros((1, y.shape[1]))
        self.n = n  # número de neurônios ocultos.
        # número de neurônios em cada camada (entrada, intermediária e saída).
        self.dims = [x.shape[0], n, 1]
        # um dicionário que conterá os parâmetros W e b entre as camadas da rede (entrada-intermediária e intermediária-saída); ou seja: W1, b1, W2 e b2.
        self.param = {}
        self.ch = {}  # parâmetro intermediário de cálculo.
        self.grad = {}  # parâmetro intermediário de cálculo.
        self.loss = []  # erro de treinamento (BCE).
        self.lr = 0.1  # taxa de aprendizagem.
        # número de observações do conjunto de treinamento.
        self.sam = y.shape[1]

    def nInit(self):  # inicializa os pesos e os bias.

        self.param['W1'] = np.random.uniform(-math.sqrt(6)/math.sqrt(
            self.dims[0]+1), math.sqrt(6)/math.sqrt(self.dims[0]+1), (self.dims[1], self.dims[0]))
        self.param['b1'] = np.random.uniform(-math.sqrt(6)/math.sqrt(
            self.dims[0]+1), math.sqrt(6)/math.sqrt(self.dims[0]+1), (self.dims[1], 1))
        self.param['W2'] = np.random.uniform(-math.sqrt(6)/math.sqrt(
            self.dims[1]+1), math.sqrt(6)/math.sqrt(self.dims[1]+1), (self.dims[2], self.dims[1]))
        self.param['b2'] = np.random.uniform(-math.sqrt(6)/math.sqrt(
            self.dims[1]+1), math.sqrt(6)/math.sqrt(self.dims[1]+1), (self.dims[2], 1))
        return

# Definindo função para cálculo de erro quadrático:
    def nloss(self):
        loss = bce(y, self.Yh)
        return loss

    def forward(self, x):
        Z1 = self.param['W1'].dot(x) + self.param['b1']
        A1 = sigmoid(Z1)
        self.ch['Z1'], self.ch['A1'] = Z1, A1
        Z2 = self.param['W2'].dot(A1) + self.param['b2']
        A2 = sigmoid(Z2)
        self.ch['Z2'], self.ch['A2'] = Z2, A2
        self.Yh = A2
# Acabamos de processar nossos dados de entrada pela rede e produzimos Yh, uma saída.
# O próximo passo é descobrir o quão bom foi o nosso resultado.
# Para isso, podemos comparar a saída que produzimos com a saída que deveríamos ter obtido: Yh e Y. Para isso, aplicaremos a função de erro:
        loss = self.nloss()
        return self.Yh, loss

    def backward(self):
        dLoss_Yh = - (y-self.Yh)

        dLoss_Z2 = dLoss_Yh * dsigmoid(self.ch['Z2'])
        dLoss_A1 = np.dot(self.param["W2"].T, dLoss_Z2)
        dLoss_W2 = 1/self.ch['A1'].shape[1] * np.dot(dLoss_Z2, self.ch['A1'].T)
        dLoss_b2 = 1/self.ch['A1'].shape[1] * \
            np.dot(dLoss_Z2, np.ones([dLoss_Z2.shape[1], 1]))

        dLoss_Z1 = dLoss_A1 * dsigmoid(self.ch['Z1'])
        dLoss_A0 = np.dot(self.param["W1"].T, dLoss_Z1)
        dLoss_W1 = 1/x.shape[1] * np.dot(dLoss_Z1, x.T)
        dLoss_b1 = 1/x.shape[1] * \
            np.dot(dLoss_Z1, np.ones([dLoss_Z1.shape[1], 1]))

        self.param["W1"] = self.param["W1"] - self.lr * dLoss_W1
        self.param["b1"] = self.param["b1"] - self.lr * dLoss_b1
        self.param["W2"] = self.param["W2"] - self.lr * dLoss_W2
        self.param["b2"] = self.param["b2"] - self.lr * dLoss_b2

    def train(self, X, Y, iter=1501):
        for i in range(0, iter):
            Yh, loss = self.forward(x)
            self.backward()
        self.loss.append(loss)
        return

    def test(self, x, y):
        pred, loss = self.forward(x)
        return loss

### Executando Método 2:

In [ ]:
# Inicializando contagem do tempo:
ini = time.time()

# Abrindo listas para Acurácia MAX (treino e teste):
acc_train_max = []
acc_test_max = []

# Inicializando loop de experimentos:
for l in range(1, 51):
    np.random.seed(l)
    train_inputs, test_inputs, train_targets, test_targets = train_test_split(
        inputs_all, targets_all, test_size=0.20, random_state=l)
    print("Experimento", l,)

    # Abrindo listas para BCE e Acurácia (treino e teste):
    bce_train = []
    bce_test = []
    acc_train = []
    acc_test = []

    # Inicializando loop para neurônios ocultos:
    for n in range(1, 251):

        # Sinalização em tela de início de rede com n neurônios ocultos:
        print("Network with", n, "hidden units:")

        # Definindo entrada e saída de treino:
        x = train_inputs.T
        y = np.ndarray.tolist(train_targets)
        y = [y]
        y = np.asarray(y)

        # Criando modelo com n neurônios ocultos:
        Modelo = Method2(x, y, n)

        # Inicializando pesos:
        Modelo.nInit()

        # Treinando modelo:
        Modelo.train(x, y)

        # Registrando e exibindo BCE de treino:
        bce_train.append(Modelo.loss)
        print("Train BCE:", Modelo.loss)

        # Registrando e exibindo Acurácia de treino:
        for n in range(0, Modelo.Yh.shape[1]):
            if Modelo.Yh[0][n] > 0.5:
                Modelo.Yh[0][n] = 1
            else:
                Modelo.Yh[0][n] = 0
        Yh = [item for elem in Modelo.Yh for item in elem]
        y = np.ndarray.tolist(train_targets)
        acc_train.append(accuracy_score(y, Yh))
        print("Train Accuracy:", accuracy_score(y, Yh))

        # Definindo entrada e saída de teste:
        x = test_inputs.T
        y = np.ndarray.tolist(test_targets)
        y = [y]
        y = np.asarray(y)

        # Testando modelo e registrando BCE de teste:
        bce_test.append(Modelo.test(x, y))

        # Registrando e exibindo Acurácia de teste:
        for n in range(0, Modelo.Yh.shape[1]):
            if Modelo.Yh[0][n] > 0.5:
                Modelo.Yh[0][n] = 1
            else:
                Modelo.Yh[0][n] = 0
        Yh = [item for elem in Modelo.Yh for item in elem]
        y = np.ndarray.tolist(test_targets)
        acc_test.append(accuracy_score(y, Yh))
        print("Test Accuracy:", accuracy_score(y, Yh))
    acc_train_max.append(acc_train[acc_test.index(max(acc_test))])
    acc_test_max.append(max(acc_test))

In [ ]:
# Finalizando e exibindo contagem de tempo de processamento:
fim = time.time()
print("Training and testing time:", fim-ini, "(s)")

In [ ]:
acc_train2 = acc_train_max
acc_test2 = acc_test_max

In [ ]:
# Exibindo em tela acurácia de teste máximo médio de treino:
print("Mean of Max Train Accuracy:", np.mean(acc_train_max))
print("Standard Deviation of Max Train Accuracy:", np.std(acc_train_max))

In [ ]:
# Exibindo em tela acurácia de teste máximo médio de teste:
print("Mean of Max Test Accuracy:", np.mean(acc_test_max))
print("Standard Deviation of Max Test Accuracy:", np.std(acc_test_max))

## BOXPLOTS

### ACURÁCIA DE TREINO

In [ ]:
# Defining data:
acc_train1_100 = [x * 100 for x in acc_train1]
acc_train2_100 = [x * 100 for x in acc_train2]
data = [acc_train1_100, acc_train2_100]
fig, ax = plt.subplots()
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.1f'))
# Creating plot:
plt.boxplot(data)
# x-axis labels:
plt.xticks([1, 2], ['WILCA-BC', 'RIXM'])
# y-label:
plt.ylabel('Accuracy (%)')
# Adding title:
plt.title("Training Sample")

# show plot:
plt.show()

### ACURÁCIA DE TESTE

In [ ]:
# Defining data:
acc_test1_100 = [x * 100 for x in acc_test1]
acc_test2_100 = [x * 100 for x in acc_test2]
data = [acc_test1_100, acc_test2_100]
fig, ax = plt.subplots()
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.1f'))
# Creating plot
plt.boxplot(data)
# x-axis labels
plt.xticks([1, 2], ['WILCA-BC', 'RIXM'])
# y-label
plt.ylabel('Accuracy (%)')
# Adding title
plt.title("Test Sample")

# show plot
plt.show()

## TESTES DE NORMALIDADE (SHAPIRO-WILK)

### ACURÁCIA DE TREINO MÉTODO 1

In [ ]:
from scipy.stats import shapiro
# Shapiro-Wilk Normality Test
stat, p = shapiro(acc_train1)
print(stat, p)
if p > 0.05:
    print('Probably Gaussian')
else:
    print('Probably not Gaussian')

### ACURÁCIA DE TESTE MÉTODO 1

In [ ]:
# Shapiro-Wilk Normality Test
stat, p = shapiro(acc_test1)
print(stat, p)
if p > 0.05:
    print('Probably Gaussian')
else:
    print('Probably not Gaussian')

### ACURÁCIA DE TREINO MÉTODO 2

In [ ]:
# Shapiro-Wilk Normality Test
stat, p = shapiro(acc_train2)
print(stat, p)
if p > 0.05:
    print('Probably Gaussian')
else:
    print('Probably not Gaussian')

### ACURÁCIA DE TESTE MÉTODO 2

In [ ]:
# Shapiro-Wilk Normality Test
stat, p = shapiro(acc_test2)
print(stat, p)
if p > 0.05:
    print('Probably Gaussian')
else:
    print('Probably not Gaussian')

## TESTES DE SIMILARIDADE

### NORMAL DISTRIBUTIONS (STUDENT'S T-TEST)

### ACURÁCIA DE TREINO

In [ ]:
# Student's t-test
from scipy.stats import ttest_ind
stat, p = ttest_ind(acc_train1, acc_train2)
print(stat, p)
if p > 0.05:
    print('Probably the same distribution')
else:
    print('Probably different distributions')

### ACURÁCIA DE TESTE

In [ ]:
# Student's t-test
stat, p = ttest_ind(acc_test1, acc_test2)
print(stat, p)
if p > 0.05:
    print('Probably the same distribution')
else:
    print('Probably different distributions')

### NON-NORMAL DISTRIBUTIONS (KRUSKAL-WALLIS H TEST)

### ACURÁCIA DE TREINO

In [ ]:
# Kruskal-Wallis H Test
from scipy.stats import kruskal
stat, p = kruskal(acc_train1, acc_train2)
print(stat, p)
if p > 0.05:
    print('Probably the same distribution')
else:
    print('Probably different distributions')

### ACURÁCIA DE TESTE

In [ ]:
# Kruskal-Wallis H Test
stat, p = kruskal(acc_test1, acc_test2)
print(stat, p)
if p > 0.05:
    print('Probably the same distribution')
else:
    print('Probably different distributions')

In [ ]:
acc_train1

In [ ]:
acc_test1

In [ ]:
acc_train2

In [ ]:
acc_test2

### FIM DO CÓDIGO